In [ ]:
import pandas as pd
import numpy as np
import lightgbm as lgb
import logging
from pathlib import Path

# Use repository-relative data folder instead of Google Drive
BASE_DIR = Path('..') / 'data'


In [ ]:
import pandas as pd
import numpy as np
import lightgbm as lgb
from pathlib import Path

# Use repository-relative data folder instead of Google Drive
BASE_DIR = Path('..') / 'data'

def calculate_pinball_loss(y_true, y_pred, tau=0.95):
    err = y_true - y_pred
    # WHAT: Calculate Pinball Loss | WHY: Assesses quantile regression performance by penalizing under/over predictions asymmetrically.
    return np.mean(np.where(err >= 0, tau * err, (tau - 1) * err))

def prep_holidays(holiday_path):
    df = pd.read_csv(holiday_path)
    # WHAT: Convert to datetime | WHY: Enables extraction of dt.year and dt.month.
    df['Date'] = pd.to_datetime(df['Date'])
    df['Year'] = df['Date'].dt.year
    df['Month'] = df['Date'].dt.month
    return df.groupby(['Year', 'Month']).size().reset_index(name='holiday_count')

# ==========================================
# DIRECT EXECUTION
# ==========================================

print("Loading Unified Feature Matrix...")
df = pd.read_csv(BASE_DIR / 'gold' / 'master_training_matrix.csv')

last_year = df['Year'].max()
last_month = df[df['Year'] == last_year]['Month'].max()

print("Extracting peak historical capacity volumes per outlet...")
historical_max = df.groupby('Outlet_ID')['monthly_volume'].max().reset_index()
historical_max.rename(columns={'monthly_volume': 'Historical_Max_Volume'}, inplace=True)

cat_cols = ['Outlet_Size', 'Outlet_Type', 'Month']
for col in cat_cols:
    # WHAT: Cast to category | WHY: Required for LightGBM to use its native categorical split algorithms.
    df[col] = df[col].astype('category')

cat_structures = {col: df[col].cat.categories for col in cat_cols}

val_mask = (df['Year'] == last_year) & (df['Month'] == last_month)
train_df = df[~val_mask].copy()
val_df = df[val_mask].copy()

print(f"Training on historical records up to {last_year}-{last_month - 1}")
print(f"Validating on holdout: {last_year}-{last_month} ({len(val_df)} records)")

drop_cols = ['Outlet_ID', 'monthly_volume', 'monthly_bill_value', 'Year']
features = [c for c in df.columns if c not in drop_cols]

X_train, y_train = train_df[features], train_df['monthly_volume']
X_val, y_val = val_df[features], val_df['monthly_volume']

TAU = 0.95
params = {
    'objective': 'quantile',
    'alpha': TAU,
    'metric': 'quantile',
    'learning_rate': 0.05,
    'num_leaves': 45,
    'min_data_in_leaf': 100,
    'feature_fraction': 0.8,
    'verbose': -1,
    'n_jobs': -1
}

print("Training LightGBM Quantile Model (alpha=0.95)...")
# WHAT: Create lgb.Dataset | WHY: Optimized structure needed for model training; categorical_feature flags non-numeric columns.
train_data = lgb.Dataset(X_train, label=y_train, categorical_feature=cat_cols)
val_data = lgb.Dataset(X_val, label=y_val, reference=train_data, categorical_feature=cat_cols)

model = lgb.train(
    params,
    train_data,
    num_boost_round=300,
    valid_sets=[val_data],
    callbacks=[lgb.early_stopping(stopping_rounds=30, verbose=False)]
)

val_preds = model.predict(X_val)
pinball = calculate_pinball_loss(y_val.values, val_preds, tau=TAU)
print(f"Validation Pinball Loss: {pinball:.4f}")

capture_rate = np.mean(val_preds >= y_val.values) * 100
print(f"Validation Capture Rate: {capture_rate:.1f}%")

safe_actuals = np.where(y_val.values == 0, 1e-5, y_val.values)
uplift_ratio = np.median(val_preds / safe_actuals)
print(f"Median Uplift Ratio: {uplift_ratio:.2f}x")

val_df['Predicted_Ceiling'] = val_preds
val_export = val_df[['Outlet_ID', 'monthly_volume', 'Predicted_Ceiling']]
val_export.to_csv(BASE_DIR / 'gold' / 'validation_results.csv', index=False)
print("Validation performance trace exported to '../data/gold/validation_results.csv'.")

print("Constructing January 2026 target matrix...")
raw_df = pd.read_csv(BASE_DIR / 'gold' / 'master_training_matrix.csv')
static_cols = ['Outlet_ID'] + [c for c in features if c not in ['holiday_count', 'Month']]

jan_2026_df = raw_df[static_cols].drop_duplicates(subset=['Outlet_ID'], keep='last').copy()
jan_2026_df['Year'] = 2026
jan_2026_df['Month'] = 1

holidays = prep_holidays(str(BASE_DIR / 'bronze' / 'holiday_list.csv'))
# WHAT: Left merge holidays | WHY: Ensures all outlets get the Jan 2026 holiday count attached.
jan_2026_df = pd.merge(jan_2026_df, holidays, on=['Year', 'Month'], how='left')
jan_2026_df['holiday_count'] = jan_2026_df['holiday_count'].fillna(0)

for col in cat_cols:
    # WHAT: Map categorical categories | WHY: Guarantees inference data has the exact same categorical structure/encoding as training.
    jan_2026_df[col] = pd.Categorical(jan_2026_df[col], categories=cat_structures[col])

X_inference = jan_2026_df[features]

print("Generating raw model predictions for January 2026...")
jan_2026_df['Predicted_Raw'] = model.predict(X_inference)

print("Executing empirical bounding checks against all-time historical peaks...")
jan_2026_df = pd.merge(jan_2026_df, historical_max, on='Outlet_ID', how='left')

# WHAT: np.maximum | WHY: Caps the lower bound of the prediction so we don't forecast a ceiling lower than what has already been achieved historically.
jan_2026_df['Maximum_Monthly_Liters'] = np.maximum(
    jan_2026_df['Predicted_Raw'],
    jan_2026_df['Historical_Max_Volume'].fillna(0)
)

submission = jan_2026_df[['Outlet_ID', 'Maximum_Monthly_Liters']]
output_path = BASE_DIR / 'gold' / 'fih_predictions.csv'

# WHAT: index=False | WHY: Saves to Drive without outputting pandas' arbitrary row numbers as a separate column.
submission.to_csv(output_path, index=False)
print(f"Execution complete. Final submission binary saved to: {output_path}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Loading Unified Feature Matrix...
Extracting peak historical capacity volumes per outlet...
Training on historical records up to 2025-11
Validating on holdout: 2025-12 (12484 records)
Training LightGBM Quantile Model (alpha=0.95)...
Validation Pinball Loss: 15.3528
Validation Capture Rate: 94.7%
Median Uplift Ratio: 1.84x
Validation performance trace exported to '../data/validation_results.csv'.
Constructing January 2026 target matrix...
Generating raw model predictions for January 2026...
Executing empirical bounding checks against all-time historical peaks...
Execution complete. Final submission binary saved to: /content/drive/MyDrive/data_storm/data/gold/fih_predictions.csv
